# 第 41 课｜Plotly 交互式 K 线与成交量

本 Notebook 对应「365 天量化金融」第 41 课。课程方法来自本地完整视频、课件截图和本地 Whisper 口播整理；这里将课程 API 重建为可离线执行、可核对的量化教学实现，并补充数据契约、断言与方法边界。

- 图文口播：`/Users/mojiawen/Documents/学习资料归档/_暂存/pam-video-captures/365-days-quant-finance/41_youtube-rdeQXve2IME/index.html`
- 环境：`pkm-quant`
- 数据：固定随机种子生成的离线模拟行情（**非真实股票数据**）
- 状态：可运行学习稿；不代表策略收益承诺

> **来源边界**：课程材料没有提供可独立核验的完整源码和真实行情快照。本 Notebook 复现课程方法，不复述视频中真实股票的价格结论。

## 1. 本节量化问题

静态图适合论文、课件和归档；研究过程中却经常需要放大异常区段、悬停核对某日 OHLCV，并让价格、均线和成交量共享时间轴。

本节建立两层用法：

1. **Plotly Express（`px`）**：高级快捷接口，一行快速检查收盘价；
2. **Graph Objects（`go`）**：由 Figure 和 Trace 逐层搭建 K 线、均线与成交量；
3. `make_subplots(shared_xaxes=True)`：让上下图共享显示坐标；
4. `write_html`：导出不依赖 Python 的自包含网页。

注意：交互性只是诊断能力，不会自动保证数据对齐、信号正确或结论可靠。

## 2. 输入、输出与验收

**输入**

- `daily_prices.csv` 中虚构资产 `EV_A` 的日线；
- 字段：`date/open/high/low/close/volume`。

**输出**

- 快捷收盘价折线；
- K 线 + 20/60 日简单移动平均线（SMA）+ 成交量双面板；
- `outputs/41_interactive_candlestick.html` 自包含交互网页。

**验收**

- 日期唯一、升序，OHLC 关系合法；
- 主图和副图使用完全相同的日期序列；
- Figure 中每个 Trace 的名称、类型和面板位置可核对；
- HTML 文件真实写出且不是空壳。

In [1]:
from pathlib import Path
import json
import os
import platform
import sys
import tempfile
import timeit

# Matplotlib 首次运行需要写字体缓存；放到可写临时目录，避免污染用户目录。
MPL_CACHE_DIR = Path(tempfile.gettempdir()) / "pkm-quant-matplotlib"
MPL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPL_CACHE_DIR))

import matplotlib.pyplot as plt
from matplotlib import font_manager
import numpy as np
import pandas as pd
import seaborn as sns

# 从本机字体中选择支持中文的字体，避免标题显示成方框。
preferred_chinese_fonts = [
    "PingFang SC",
    "Arial Unicode MS",
    "Hiragino Sans GB",
    "Heiti SC",
]
available_font_names = {font.name for font in font_manager.fontManager.ttflist}
chinese_font = next(
    (name for name in preferred_chinese_fonts if name in available_font_names),
    "DejaVu Sans",
)
plt.rcParams["font.sans-serif"] = [chinese_font, "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

def find_project_root():
    cwd = Path.cwd().resolve()
    candidates = [
        cwd,
        cwd / "实验" / "p2-python-toolchain",
        cwd / "p2-python-toolchain",
        *cwd.parents,
    ]
    for candidate in candidates:
        if (candidate / "课程辅助设计.md").exists():
            return candidate
    raise FileNotFoundError("请从 p2-python-toolchain 或量化交易项目目录启动 JupyterLab")

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data" / "fixtures"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
RANDOM_SEED = 365

print("Python:", sys.version.split()[0])
print("解释器:", sys.executable)
print("项目:", PROJECT_ROOT)
print("绘图字体:", chinese_font)

Python: 3.11.15
解释器: /opt/homebrew/Caskroom/miniforge/base/envs/pkm-quant/bin/python
项目: /Users/mojiawen/Documents/学习资料归档/_暂存/personal_knowledge_manager/量化交易/实验/p2-python-toolchain
绘图字体: PingFang SC


In [2]:
# 本课新增 Plotly；分别导入快捷接口、底层对象接口和子图工具。
import plotly
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import IFrame, display

print("Plotly:", plotly.__version__)

Plotly: 6.9.0


## 3. 先查看原始长表

`read_csv(parse_dates=["date"])` 在读取时把日期文本转换为 `datetime64`。这里先显示原始列和前几行，再筛选单只资产；不能在不知道字段语义时直接绘图。

In [3]:
# 读取完整模拟日线，并展示数据规模、资产和原始字段。
daily = pd.read_csv(DATA_DIR / "daily_prices.csv", parse_dates=["date"])

print("原始 shape:", daily.shape, "含义: 股票日 × 字段")
print("资产:", daily["symbol"].drop_duplicates().tolist())
display(daily.head(6))

assert not daily.duplicated(["date", "symbol"]).any()
assert {"date", "symbol", "open", "high", "low", "close", "volume"}.issubset(daily.columns)

原始 shape: (4536, 9) 含义: 股票日 × 字段
资产: ['BANK_A', 'INSURE_A', 'EV_A', 'POWER_A', 'ENERGY_A', 'TECH_A']


,date,symbol,sector,open,high,low,close,adj_close,volume
0,2023-01-03,BANK_A,金融,11.7051,11.7522,11.6747,11.7218,11.7218,2822797
1,2023-01-04,BANK_A,金融,11.7142,11.7878,11.6199,11.6934,11.6934,1999548
2,2023-01-05,BANK_A,金融,11.6517,11.7478,11.5905,11.6864,11.6864,1974821
3,2023-01-06,BANK_A,金融,11.6692,11.7522,11.5632,11.6461,11.6461,3999555
4,2023-01-09,BANK_A,金融,11.6141,11.6557,11.4507,11.4919,11.4919,2813450
5,2023-01-10,BANK_A,金融,11.4561,11.5439,11.3933,11.4810,11.4810,5359349


## 4. 整理 Plotly 所需 OHLCV

课程中的 Candlestick（蜡烛图）至少需要日期、开盘、最高、最低、收盘。成交量是独立 Trace。

- `set_index("date")`：日期成为行标签；
- `rename(...)`：统一为易辨认的首字母大写字段；
- `tail(160)`：只画最近 160 个交易日，避免几千根蜡烛挤成一片；
- `rolling(20/60, min_periods=...)`：只使用当日及过去收盘价。

In [4]:
# 筛选一只虚构资产并改成绘图契约；均线未满窗口时保持 NaN。
ohlcv = (
    daily.loc[
        daily["symbol"].eq("EV_A"),
        ["date", "open", "high", "low", "close", "volume"],
    ]
    .sort_values("date")
    .set_index("date")
    .rename(
        columns={
            "open": "Open",
            "high": "High",
            "low": "Low",
            "close": "Close",
            "volume": "Volume",
        }
    )
    .tail(160)
)
ohlcv["SMA20"] = ohlcv["Close"].rolling(20, min_periods=20).mean()
ohlcv["SMA60"] = ohlcv["Close"].rolling(60, min_periods=60).mean()

print("绘图区间:", ohlcv.index.min().date(), "→", ohlcv.index.max().date())
print("shape:", ohlcv.shape, "含义: 日期 × OHLCV/均线")
display(ohlcv.head(3))
display(ohlcv.tail(3))

assert isinstance(ohlcv.index, pd.DatetimeIndex)
assert ohlcv.index.is_unique and ohlcv.index.is_monotonic_increasing
assert (ohlcv["High"] >= ohlcv[["Open", "Close"]].max(axis=1)).all()
assert (ohlcv["Low"] <= ohlcv[["Open", "Close"]].min(axis=1)).all()
assert ohlcv["SMA20"].iloc[:19].isna().all()
assert ohlcv["SMA60"].iloc[:59].isna().all()

绘图区间: 2025-04-16 → 2025-11-25
shape: (160, 7) 含义: 日期 × OHLCV/均线


,Open,High,Low,Close,Volume,SMA20,SMA60
date,,,,,,,
2025-04-16,39.4591,40.6820,39.1968,40.4134,3614864,NaN,NaN
2025-04-17,40.4681,40.7702,39.6421,39.9403,3244437,NaN,NaN
2025-04-18,40.0104,40.3313,39.6766,39.9973,3395157,NaN,NaN


,Open,High,Low,Close,Volume,SMA20,SMA60
date,,,,,,,
2025-11-21,42.9352,43.2227,41.7473,42.0288,2828482,42.536300,43.618687
2025-11-24,42.0224,42.2802,41.4361,41.6918,3118668,42.509645,43.560940
2025-11-25,41.6482,41.8792,40.2380,40.4624,4571272,42.472355,43.475242


## 5. Plotly Express：快速看一眼

`px.line(data_frame, x, y)` 自动创建 Figure 和一条 Scatter Trace。它适合探索单一序列；复杂金融图仍应使用 Graph Objects。

In [5]:
# 快捷接口只画收盘价；检查生成的 Trace，而不是仅凭肉眼判断。
quick_data = ohlcv.reset_index()
quick_fig = px.line(
    quick_data,
    x="date",
    y="Close",
    title="EV_A 模拟收盘价｜Plotly Express",
    labels={"date": "日期", "Close": "模拟价格"},
)
quick_fig.update_layout(template="plotly_white", height=360)

quick_trace = quick_fig.data[0]
print("Trace 数量:", len(quick_fig.data))
print("Trace 类型:", quick_trace.type)
print("点数:", len(quick_trace.x))

assert len(quick_fig.data) == 1
assert quick_trace.type == "scatter"
assert len(quick_trace.x) == len(ohlcv)

Trace 数量: 1
Trace 类型: scatter
点数: 160


## 6. Graph Objects：先用 5 日小样本理解 Trace

`go.Candlestick` 不是直接“画一张图片”，而是创建一层可交互的图形数据：

- `x`：日期；
- `open/high/low/close`：同长度的四条数组；
- `name`：图例名称。

Figure 是容器，Trace 是图层。后续均线和成交量都是额外 Trace。

In [6]:
# 用 5 日小样本构造一层蜡烛，并逐项核对 Trace 的数据长度。
five_days = ohlcv.iloc[:5]
small_fig = go.Figure(
    data=[
        go.Candlestick(
            x=five_days.index,
            open=five_days["Open"],
            high=five_days["High"],
            low=five_days["Low"],
            close=five_days["Close"],
            name="5 日 OHLC",
        )
    ]
)
small_trace = small_fig.data[0]

trace_check = pd.DataFrame(
    {
        "date": pd.to_datetime(small_trace.x),
        "open": np.asarray(small_trace.open, dtype=float),
        "high": np.asarray(small_trace.high, dtype=float),
        "low": np.asarray(small_trace.low, dtype=float),
        "close": np.asarray(small_trace.close, dtype=float),
    }
)
display(trace_check)

assert small_trace.type == "candlestick"
assert len(trace_check) == 5
np.testing.assert_allclose(trace_check["close"], five_days["Close"])

,date,open,high,low,close
0,2025-04-16,39.4591,40.6820,39.1968,40.4134
1,2025-04-17,40.4681,40.7702,39.6421,39.9403
2,2025-04-18,40.0104,40.3313,39.6766,39.9973
3,2025-04-21,40.0733,40.4905,39.7481,40.1645
4,2025-04-22,40.1375,40.3795,38.7634,38.9986


## 7. `make_subplots`：价格与成交量共享时间轴

`shared_xaxes=True` 只让两个面板共享**显示坐标**，不会替你检查数据是否对齐。因此这里所有 Trace 都从同一个 `ohlcv` 表取日期，并在生成后再次核对。

`row_heights=[0.72, 0.28]` 控制上下高度；`vertical_spacing` 控制间距。A 股习惯常用红涨绿跌，但颜色不是跨市场通用语义，必须保留图例和字段说明。

In [7]:
# 逐层加入 K 线、两条均线和成交量，所有图层共用同一个日期索引。
fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.04,
    row_heights=[0.72, 0.28],
    subplot_titles=("模拟价格与均线", "成交量"),
)

fig.add_trace(
    go.Candlestick(
        x=ohlcv.index,
        open=ohlcv["Open"],
        high=ohlcv["High"],
        low=ohlcv["Low"],
        close=ohlcv["Close"],
        increasing_line_color="#d62728",
        decreasing_line_color="#2ca02c",
        name="OHLC",
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(x=ohlcv.index, y=ohlcv["SMA20"], mode="lines", name="SMA20"),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(x=ohlcv.index, y=ohlcv["SMA60"], mode="lines", name="SMA60"),
    row=1,
    col=1,
)

volume_colors = np.where(
    ohlcv["Close"].ge(ohlcv["Open"]),
    "#d62728",
    "#2ca02c",
)
fig.add_trace(
    go.Bar(
        x=ohlcv.index,
        y=ohlcv["Volume"],
        marker_color=volume_colors,
        name="Volume",
    ),
    row=2,
    col=1,
)

fig.update_layout(
    title="EV_A 模拟行情｜交互 K 线、均线与成交量",
    template="plotly_white",
    height=720,
    hovermode="x unified",
    legend_orientation="h",
    margin=dict(l=60, r=30, t=90, b=45),
)
fig.update_xaxes(rangeslider_visible=False, row=1, col=1)
fig.update_yaxes(title_text="模拟价格", row=1, col=1)
_ = fig.update_yaxes(title_text="成交量", row=2, col=1)

### 图层结构检查

不能因为图“看起来对齐”就默认正确。下面把 Figure 内部的 Trace 清单变成表格，并验证四层的日期与原数据完全一致。

In [8]:
# 把 Figure 内部结构转换为可核对表格；四层日期必须与 ohlcv 索引一致。
trace_manifest = pd.DataFrame(
    [
        {
            "序号": index,
            "名称": trace.name,
            "类型": trace.type,
            "点数": len(trace.x),
        }
        for index, trace in enumerate(fig.data, start=1)
    ]
)
display(trace_manifest)

expected_names = ["OHLC", "SMA20", "SMA60", "Volume"]
assert trace_manifest["名称"].tolist() == expected_names
assert trace_manifest["点数"].eq(len(ohlcv)).all()
for trace in fig.data:
    pd.testing.assert_index_equal(
        pd.DatetimeIndex(trace.x),
        ohlcv.index,
        check_names=False,
    )

,序号,名称,类型,点数
0,1,OHLC,candlestick,160
1,2,SMA20,scatter,160
2,3,SMA60,scatter,160
3,4,Volume,bar,160


## 8. 导出自包含交互网页

`include_plotlyjs=True` 会把 Plotly JavaScript 一并写入文件，所以网页较大，但离线双击也能缩放和悬停。`write_image` 是另一条静态导出链路，通常还需 Kaleido；本课不混用。

In [9]:
# 写出自包含 HTML，并检查文件存在、大小和关键 HTML 标记。
interactive_path = OUTPUT_DIR / "41_interactive_candlestick.html"
fig.write_html(
    interactive_path,
    include_plotlyjs=True,
    full_html=True,
)
html_head = interactive_path.read_text(encoding="utf-8")[:500]

print("已写入:", interactive_path)
print("文件大小:", round(interactive_path.stat().st_size / 1024 / 1024, 2), "MiB")

assert interactive_path.exists()
assert interactive_path.stat().st_size > 1_000_000
assert "<html" in html_head.lower()
assert "plotly" in html_head.lower()

已写入: /Users/mojiawen/Documents/学习资料归档/_暂存/personal_knowledge_manager/量化交易/实验/p2-python-toolchain/outputs/41_interactive_candlestick.html
文件大小: 4.67 MiB


### 在 Notebook / 静态报告中嵌入交互成品

下方 iframe 指向 `outputs/41_interactive_candlestick.html`。在 Jupyter 和导出的课程报告里都可直接操作；若单独移动报告文件，应同时保留 `outputs/` 的相对目录结构。

In [10]:
# 相对路径同时适用于 notebooks/ 和 reports/ 目录中的页面。
display(
    IFrame(
        src="../outputs/41_interactive_candlestick.html",
        width="100%",
        height=740,
    )
)

## 9. 量化意义与方法边界

- 交互图适合异常日复核、策略诊断和研究分享；静态图适合课件、PDF 和版本归档。
- K 线必须使用同一频率、同一时区、同一复权口径的 OHLC；不能把复权收盘与未复权开高低混在一根蜡烛中。
- `shared_xaxes=True` 不会检查数据索引；绘图前仍需显式对齐。
- SMA 是历史平滑指标，不是“平均持仓成本”的严格估计，也不保证穿越后继续上涨。
- 成交量放大不能仅凭 OHLCV 判断“主力买入”或交易者身份。
- 自包含 HTML 会携带完整悬停数据；真实项目发布前要检查敏感数据、版权和文件体积。
- 交互图帮助发现问题，不替代样本外验证、成本模型和统计检验。

## 10. 可复用结论与练习

可复用流程：

```text
OHLCV 契约 → 日期/字段断言 → 指标计算
           → Figure + Trace 分层 → 图层清单校验
           → 自包含 HTML → 浏览器交互复核
```

练习：

1. 把 `EV_A` 改为另一只模拟资产，仍保持四层日期完全一致。
2. 增加策略净值和回撤面板，但不要使用双轴制造视觉相关。
3. 将 160 日改为 756 日，比较可读性和 HTML 文件大小。
4. 故意打乱成交量日期，验证现有断言能够发现错位。